# 第6回課題：CNNで文字認識をしよう

最も基本的な画像認識タスクとして、手書き文字（数字）認識をやってみましょう。   
深層学習による画像認識のためのアーキテクチャであるCNNの実装を使います。   
演習で配ったCIFAR-10のコードを参考にしてください。

<H2><font color="red">課題を提出する際は、画像が表示されている状態で保存し、このmp_ex6.ipynbファイルのみを提出してください。</font></H2>
<H2><font color="red">中間結果を出力しないでください。ファイルサイズは1MB以内とします。</font></H2>

## 今回挑戦する画像認識タスク
小規模な画像認識タスクとして、手書き文字のデータセット[MNIST(Mixed National Institute of Standards and Technology database)](http://yann.lecun.com/exdb/mnist/)を使って、０から９までの10種類の記号に対する文字認識を行います。   
MNISTは1枚あたり28x28ピクセルで、訓練データとして60,000枚、評価データとして10,000枚の画像が与えられています。    

## 利用するライブラリ
本プログラムでは、ニューラルネットワーク実装のためのライブラリとして、Tensorflowを利用しています。   

+ Tensorflow 2.7 Documentation: https://www.tensorflow.org/api_docs/python/tf

## 実行環境：Colaboratoryを強く推奨します

CPUでも実行できると思いますが、モデルの学習に計算負荷がかかります。   
Colaboratoryで「ランタイム＞ランタイムのタイプを変更」で「ハードウェアアクセラレータ」をGPUにしてから実行することをお勧めします。

# 1. ライブラリのインストール

本プログラムでは、ニューラルネットワーク実装のためのライブラリとして、Tensorflowを利用しているので、'tensorflow'をインストールします。   

**Colaboratoryの使用を強くお勧めします。**

## 1.1 ローカルPCの場合（非推奨）

**必ず仮想環境を作ってからパッケージをインストールしてください。**   
ガイダンスの環境設定の資料を参照して、ライブラリのインストールをお願いします。
1. Anaconda Navigatorを開く
2. 「Environments」のタブを開き、中央のフレームで「base(root)」とある右側の「▶」をクリックし、"Open Terminal"をクリック
3. コマンドプロンプトから以下の二つのコマンドを実行  

``conda install -c anaconda tensorflow==2.20.0``   

**トラブルが起きる場合はColaboratoryをご利用ください。**

## 1.2 Colaboratoryの場合（推奨）
Colaboratoryではtensorflowが標準でインストール済みのため、コマンドを実行する必要はありません。

## ライブラリを読み込み

今回使うライブラリであるTensorflowをインポートしましょう。

In [ ]:
import os
import numpy as np
import tensorflow as tf

# 計算するたびに違う答えにならないよう、ランダムシードを設定する
np.random.seed(seed=0)

# 2. 学習データの準備

## MNISTデータセットをダウンロード
MNIST datasetsは様々な場面で使われているので、Kerasのライブラリの中に、データセットをダウンロードするための関数が用意されています。   
これを使ってデータセットをダウンロードし、読み込みましょう。   
データサイズは11MBです。    
PCで以下のセルを実行すると、ダウンロードされたファイルがホームディレクトリの下の'.keras/datasets'の下に保存されています。   
次回から同じPC＋アカウントでこのコマンドを実行する際は、改めてダウンロードされることはありません。   
（Colaboratoryはランタイムを終了するとファイルがすべて消去されるため、毎回ダウンロードすることになります）

In [ ]:
# the data, split between train and test sets
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

入力画像を確認してみましょう。

x_trainは画像の集合で、 (60,000枚) x (28 pixel) x (28 pixel)のテンソルです。
1枚目の画像サイズを見ると、28x28であることがわかります。
MNISTはカラー画像ではなくグレースケール画像なので、チャネルは1です（カラーの場合はRGBの2チャネルでしたね）。

In [ ]:
print('x_train shape:', x_train.shape)
print(x_train.shape[0], 'train samples')
print(x_test.shape[0], 'test samples')

また、y_trainには60,000枚の各画像の正解のラベルが入っています。   

In [ ]:
print(len(y_train)) # 50,000枚の画像のラベルが入っている
print(y_train[:10]) # 最初の10枚のラベル（整数値で記録されている）

訓練画像を1枚表示させてみましょう。非常に画素が粗い画像であることがわかります。

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# 画像をいろいろと入れ替えてみましょう。
id =7
plt.imshow(x_train[id], cmap = "gray")
plt.axis('off')

print('Class label: ', y_train[id])

画像データの形を見てみましょう。

In [ ]:
print(x_train.shape)

CIFAR-10はカラー画像だったので、入力した時点でテンソルでしたが、グレースケール画像はチャネルが１なので、テンソルに変換する必要があります。

In [ ]:
x_train = x_train.reshape(x_train.shape[0], x_train.shape[1], x_train.shape[2], 1)
x_test = x_test.reshape(x_test.shape[0], x_train.shape[1], x_train.shape[2], 1)

次元が1つ増えました。

In [ ]:
print(x_train.shape)

## 正解ラベルをone-hotベクトルに変換
`y_train`や`y_test`は、そのアドレスに対応する画像の正解ラベルが0から9までの整数値で記録されています。   
一方、1枚の入力画像に対してCNNが最終層で出力するのは、その画像が0から9までの各クラスである尤もらしさ（確率）ですから、それに対応するように、正解ラベルもone-hotベクトル（すなわち、正解のクラスだけが1、残りが0のベクトル）に変換しましょう。   

クラスは全部で10クラスですから、10次元のone-hotベクトルに変換します。   
たとえば正解が「６」であるような画像のクラスは"6"ですが、これを10次元のone-hotベクトルに変換すると、6番目だけが1で残りは0であるような`[0 0 0 0 0 0 1 0 0 0]`というベクトルに変換されます。   
これを、すべての画像（計60,000枚）について行うので、`y_train`は60,000x10の行列になります。

以下のコードの?に適切な数字を入れてください。

In [ ]:
# convert class vectors to binary class matrices
num_classes = 10
y_train = tf.keras.utils.to_categorical(y_train, num_classes)
y_test = tf.keras.utils.to_categorical(y_test, num_classes)

y_trainのサイズと、最初の10枚分の中身を見てみましょう。

In [ ]:
print('y_train shape:', y_train.shape)
print(y_train[:10])

## 画像の濃淡値を整数値

各画素の濃淡値は0～255までの整数値で記録されているので、255で割って0～1までの値に正規化します。

In [ ]:
x_train = x_train.astype('float32')
x_test = x_test.astype('float32')
x_train /= 255
x_test /= 255

# 3. ネットワークを設計

以下のようなネットワークになるよう、モデルを設計してください。   
モデルのデザインはコードセルのコメントに従ってください。   
また、コードは'ImageRecognition3.ipynb'を参考にしてください。   
```
Model: "sequential"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 24, 24, 32)     │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 22, 22, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 22, 22, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 3136)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       401,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 10)             │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘
 Total params: 422,154 (1.61 MB)
 Trainable params: 422,154 (1.61 MB)
 Non-trainable params: 0 (0.00 B)

```


In [ ]:
model = tf.keras.models.Sequential()
# Convolution 1 フィルタ32枚、各フィルタのカーネルサイズ5x5 ストライドはデフォルト (1, 1)、パディングはvalid


# Convolution 2 フィルタ64枚、各フィルタのカーネルサイズ3x3 ストライドはデフォルト (1, 1)、パディングはvalid


# 'relu'で活性化


# Max Pooling、カーネルサイズは 3x3、パディングはvalid


# 25%のユニットをドロップアウト


# テンソルを一列のベクトルに平坦化


# Full Connection 1 # ユニット数はベクトルサイズと同じ128


# 'relu'で活性化


# 30%のユニットをドロップアウト


# Full Connection 2 # ユニット数はクラス数と同じ10


# 最後の活性化関数は出力を確率にするためsoftmaxを使用



ネットワーク構造のサマリを出力してみましょう。

In [ ]:
model.summary()

# 4. 学習
## 学習パラメータ設定

以下のパラメータを設定してください。
*   バッチサイズは32
*   エポック数は10

In [ ]:
batch_size = ?
epochs = ?

## 最適化手法・損失関数・評価関数の設定
最適化手法を選択します。   

- 最適化手法は'AdamW'を使ってください。パラメータは学習率を0.0001、`weight_decay`を0.005とします。
- 判別問題として解いてください
- 精度は'accuracy'(正解率）で評価してください

In [ ]:
model.compile(loss='categorical_crossentropy',
              optimizer=?
              metrics=?
              )

## 学習

上で設計したネットワークに訓練データを与えてモデルを学習します。   
**時間がかかりますので覚悟してください（お手持ちのPCに不安がある方は、Google Colaboratoryをご利用ください）。**


In [ ]:
history = model.fit(x_train, y_train,
          batch_size=batch_size,
          epochs=epochs,
          verbose=1,
          validation_data=(x_test, y_test))

## 4. モデルの保存と評価

モデルの学習には長い時間がかかりますので、せっかく学習したモデルは保存しておきましょう。   
Colaboratoryで実行している場合は、保存後にPCにダウンロードしておかないと、ランタイムを停止する際に削除されてしまうのでご注意ください。

In [ ]:
save_dir = os.path.join(os.getcwd(), 'saved_models')  # モデルの保存先
model_name = 'keras_cifar10_trained_model.keras'      # ← 拡張子を付ける
if not os.path.isdir(save_dir):
    os.makedirs(save_dir)
model_path = os.path.join(save_dir, model_name)

model.save(model_path)  # Kerasのネイティブ形式で保存
print(f'Saved trained model at {model_path}')

正解率を評価します。

In [ ]:
score = model.evaluate(x_test, y_test, verbose=1)
print('Test loss:', score[0])
print('Test accuracy:', score[1])

モデル学習における訓練データと評価データのロスの下がり方、および正解率の上がり方を可視化してみましょう。   
まだ収束していないので、エポック数を増やしたほうがいいと言えます。

In [ ]:
import matplotlib.pyplot as plt

# history オブジェクトから訓練データのロスの値を取得
training_loss = history.history['loss']
# history オブジェクトから訓練データの正解率の値を取得
training_accuracy = history.history['accuracy']
# history オブジェクトから評価用データのロスの値を取得
test_loss = history.history['val_loss']
# history オブジェクトから評価用データの正解率の値を取得
test_accuracy = history.history['val_accuracy']

# x軸のラベルはエポック数
x_labels = range(1, epochs + 1)

# 2つのy軸を持つグラフを作成
fig, ax1 = plt.subplots()

# ロスを左側のy軸にプロット
ax1.plot(x_labels, training_loss, 'b-', label='Training Loss')
ax1.plot(x_labels, test_loss, 'b--', label='Test Loss')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Loss', color='b')
ax1.tick_params('y', colors='b')
ax1.legend(loc='center right')

# 精度を右側のy軸にプロット
ax2 = ax1.twinx()
ax2.plot(x_labels, training_accuracy, 'r-', label='Training Accuracy')
ax2.plot(x_labels, test_accuracy, 'r--', label='Test Accuracy')
ax2.set_ylabel('Accuracy', color='r')
ax2.tick_params('y', colors='r')
ax2.legend(loc='lower left')

plt.title('Training Loss and Accuracy')
plt.grid()
plt.show()

# 入力データに対するクラスラベルの予測


どの画像がどのように判定されたか見てみましょう。  


In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
# idをいろいろ入れ替えて、正解と予測を見比べてみましょう。
id = 3
plt.imshow(np.squeeze(x_test[id]), cmap='gray', vmin=0, vmax=1)
ans = np.argmax(y_test[id])
plt.axis('off')

target = x_test[id] # 単体の入力データを用意
predict_prob=model.predict(np.array([target]))
predict_class=np.argmax(predict_prob,axis=1)
print(predict_class)
print('Truth: ', ans, '\tPredicted: ', predict_class[0])

# 参考文献

MNIST: [Learning Multiple Layers of Features from Tiny Images, Alex Krizhevsky, 2009.](https://www.cs.toronto.edu/~kriz/learning-features-2009-TR.pdf)  

<!-- label:2026S2 -->